In [2]:
from pyspark.sql import SparkSession

In [3]:
jar_files = [
    "/Users/ashwinkarki/Documents/Clickhouse/chapter-8-Spark/clickhouse-jdbc-0.9.2.jar"  # absolute path
]

In [4]:
spark = SparkSession.builder \
    .appName("example") \
    .master("local") \
    .config("spark.jars", ",".join(jar_files)) \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/18 13:50:51 WARN Utils: Your hostname, Ashwins-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.157 instead (on interface en0)
26/02/18 13:50:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/02/18 13:50:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [5]:
spark

In [6]:
from pyspark.sql.functions import rand

df=spark.range(1,100001) \
.withColumnRenamed("id","user_id") \
.withColumn("score",(rand()*100).cast("int")) \
.withColumn("country",(rand()*3).cast("int" ))                

In [7]:
df.show(5)

+-------+-----+-------+
|user_id|score|country|
+-------+-----+-------+
|      1|   95|      2|
|      2|   85|      1|
|      3|   24|      0|
|      4|   58|      0|
|      5|   42|      0|
+-------+-----+-------+
only showing top 5 rows


In [8]:
from pyspark.sql.functions import when

df=df.withColumn("country",when(df.country==0,"Nepal").when(df.country==1,"India").otherwise("USA"))

In [9]:
df.show()

+-------+-----+-------+
|user_id|score|country|
+-------+-----+-------+
|      1|   95|    USA|
|      2|   85|  India|
|      3|   24|  Nepal|
|      4|   58|  Nepal|
|      5|   42|  Nepal|
|      6|   82|  India|
|      7|    0|  India|
|      8|   15|  Nepal|
|      9|   34|  Nepal|
|     10|   35|  Nepal|
|     11|   91|    USA|
|     12|   18|  India|
|     13|   50|    USA|
|     14|   11|  Nepal|
|     15|    2|  Nepal|
|     16|   13|  Nepal|
|     17|   21|  India|
|     18|   74|  India|
|     19|   46|  Nepal|
|     20|   46|  Nepal|
+-------+-----+-------+
only showing top 20 rows


In [ ]:
df.write \
    .format("jdbc") \
    .option("url", "jdbc:clickhouse://ddutaqmpsi.us-west-2.aws.clickhouse.cloud:8443/users_db?ssl=true") \
    .option("dbtable", "users") \
    .option("user", "default") \
    .option("password", "") \
    .option("driver", "com.clickhouse.jdbc.ClickHouseDriver") \
    .option("batchsize", 100000) \
    .mode("append") \
    .save()


26/02/18 13:52:12 WARN ClientConfigProperties: Unknown and unmapped config properties: {ssl=true}
26/02/18 13:52:15 WARN ClientConfigProperties: Unknown and unmapped config properties: {ssl=true}
26/02/18 13:52:15 WARN JdbcUtils: Requested isolation level 1, but transactions are unsupported
26/02/18 13:52:20 WARN ClientConfigProperties: Unknown and unmapped config properties: {ssl=true}


Py4JJavaError: An error occurred while calling o116.save.
: org.apache.spark.SparkSQLException: [UNRECOGNIZED_SQL_TYPE] Unrecognized SQL type - name: UInt64, id: OTHER. SQLSTATE: 42704
	at org.apache.spark.sql.errors.QueryExecutionErrors$.unrecognizedSqlTypeError(QueryExecutionErrors.scala:1032)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcUtils$.getCatalystType(JdbcUtils.scala:245)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcUtils$.$anonfun$getSchema$1(JdbcUtils.scala:330)
	at scala.Option.getOrElse(Option.scala:201)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcUtils$.getSchema(JdbcUtils.scala:330)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCRDD$.$anonfun$getQueryOutputSchema$3(JDBCRDD.scala:91)
	at scala.util.Using$.resource(Using.scala:296)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCRDD$.$anonfun$getQueryOutputSchema$2(JDBCRDD.scala:89)
	at scala.util.Using$.resource(Using.scala:296)
	at org.apache.spark.sql.execution.dataso